# Part 5, notebook 4 of 6: is the sandbox transparent?

A sandbox that changes what the code does would be useless: you would be measuring the sandbox, not the model. Part 4 recorded every cell the model wrote (six runs, 72 episodes). This notebook replays each recorded cell through a **fresh container per episode** and compares the result with what the unsandboxed Jupyter kernel returned at the time.

**The analogy.** Same exam answers, handed to a different examiner in a different room. If the marks differ, the room mattered.

No model is involved: the cells are the recorded ones. Compared per cell: stdout (first 2000 characters), whether it raised an error, the submission, and the number of pictures.

In [1]:
import sys, json, re, time
from pathlib import Path
sys.path.insert(0, "../minimal_loop")
import numpy as np
from docker_session import DockerSession

ROOT = Path("..").resolve()
images = np.load(ROOT.parent / "agentic-factory-segmentation/data/void_dataset_v2/val.npz")["images"]
COND = {"A": (False, False), "B": (True, False), "C": (True, True)}   # A: no pictures, B: pictures, C: pictures + tools
print("DEV images:", images.shape, "| conditions: A = code only, B = + show(), C = + show() and mask tools")

DEV images: (600, 224, 224) | conditions: A = code only, B = + show(), C = + show() and mask tools


In [2]:
def replay(tag):
    run = json.load(open(ROOT / f"out/run_dev4_{tag}.json"))
    show, tools = COND[tag[0]]
    n = same = 0; diffs = []
    for t in run["traces"]:
        i = int(t["image"].split("_")[1])
        s = DockerSession(images[i], timeout_sec=60, with_show=show, with_tools=tools)
        try:
            for c in t["cells"]:
                if not c["code"]:
                    continue
                r = s.run(c["code"])
                sub_rec = re.search(r"\[submitted\] (\{.*\})", c["stdout"])
                ok_out = r.stdout[:2000] == c["stdout"]
                ok_err = bool(r.error) == bool(c["error"])
                ok_sub = (r.submission is not None) == bool(sub_rec) and (r.submission is None or json.dumps(r.submission, sort_keys=True) == json.dumps(json.loads(sub_rec.group(1)), sort_keys=True))
                ok_pic = len(r.images) == len(c.get("images", []))
                n += 1
                if ok_out and ok_err and ok_sub and ok_pic:
                    same += 1
                else:
                    diffs.append((tag, t["image"], c["step"], ok_out, ok_err, ok_sub, ok_pic))
        finally:
            s.close()
    return len(run["traces"]), n, same, diffs

In [3]:
t0 = time.time(); tot = same_tot = 0; all_diffs = []
for tag in ("A", "Ar2", "B", "Br2", "C", "Cr2"):
    ep, n, same, diffs = replay(tag)
    tot += n; same_tot += same; all_diffs += diffs
    print(f"{tag:4s} episodes={ep} cells={n} identical={same}", flush=True)
print(f"\nTOTAL cells replayed: {tot}; identical in the sandbox: {same_tot}; different: {tot - same_tot}   ({time.time() - t0:.0f}s)")
for d in all_diffs[:10]:
    print("DIFF (tag, image, step, stdout, error, submission, pictures):", d)

A    episodes=12 cells=140 identical=140


Ar2  episodes=12 cells=129 identical=129


B    episodes=12 cells=100 identical=100


Br2  episodes=12 cells=92 identical=92


C    episodes=12 cells=94 identical=94


Cr2  episodes=12 cells=79 identical=79



TOTAL cells replayed: 634; identical in the sandbox: 634; different: 0   (47s)


**Reading the result.** The post reports 634 of 634 identical *after one fix*: the first replay found 1 difference in 634 cells, which was a real bug in my runner: the model's cell wrote `__builtins__.round`. In a notebook `__builtins__` is a module, but in my first exec namespace it was a dict, so the cell raised `AttributeError`. The runner now sets `__builtins__` to the module (see the `NS = {...}` line in notebook 05a). That first attempt is kept in `outputs/part5_replay_attempt1.txt`.

**What this proves and what it does not.** The cell *behaviour* is the same for the cells these models actually wrote. It says nothing about cells a different model, or a prompt-injected one, might write. That is what notebook 05e is for.